In [ ]:
from langchain_ollama import ChatOllama

OLLAMA_BASE_URL = "http://192.168.1.120:11434"
MODEL_NAME = "llama3.1:8b"

query = "What is the capital of France?"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_BASE_URL,
    temperature=0.7
)

response = llm.invoke(query)
print(response.content)

The capital of France is Paris.


In [3]:
from typing import TypedDict, Annotated
import operator

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama

In [4]:
class AgentState(TypedDict):
    query: str # immutable input
    messages: Annotated[list[str], operator.add] # grows over time (reducer = +)
    answer: str # answer from the LLM
    approved: bool # controls routing (human in the loop)

In [10]:
def llm_node(state: AgentState):
    respone = llm.invoke(state["query"])
    return {
        "messages": [respone.content],
        "answer": respone.content
    }

In [ ]:
def judge_node(state: AgentState):
    approved = len(state["answer"]) < 100
    return {
        "approved": approved
    }

In [12]:
def human_review_node(state: AgentState):
    print("\n--- HUMAN REVIEW REQUIRED ---")
    print("Proposed answer:\n", state["answer"])
    decision = input("Approve? (y/n): ").lower().strip()

    return {
        "approved": decision == "y"
    }

In [ ]:
def finalize_node(state: AgentState):
    print("\n✅ FINAL ANSWER:")
    print(state["answer"])
    return {}

In [14]:
def route_after_judge(state: AgentState):
    if state["approved"]:
        return "final"
    return "human"

In [17]:
graph = StateGraph(AgentState)

graph.add_node("llm", llm_node)
graph.add_node("judge", judge_node)
graph.add_node("human_review", human_review_node)
graph.add_node("finalize", finalize_node)

graph.add_edge(START, "llm")
graph.add_edge("llm", "judge")
graph.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "final": "finalize",
        "human": "human_review"
    }
)
graph.add_edge("human_review", "finalize")
graph.add_edge("finalize", END)


In [25]:
checkpointer = MemorySaver()
app = graph.compile()

In [26]:
initial_state = {
    "query": "What is the capital of France?",
    "messages": [],
    "answer": "",
    "approved": False
}

result = app.invoke(initial_state)


✅ FINAL ANSWER:
The capital of France is Paris.


In [27]:
result

{'query': 'What is the capital of France?',
 'messages': ['The capital of France is Paris.'],
 'answer': 'The capital of France is Paris.',
 'approved': True}